<a href="https://colab.research.google.com/github/miguelloeza210/UAVLogViewer/blob/master/bitesafe.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Environment Setup

In [ ]:
import sys
import os
import torch

if 'google.colab' in sys.modules:
    print("Running in Google Colab. Setting up GitHub and Drive...")

    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive')

    ASSETS_DIR = '/content/drive/MyDrive/BiteSafe/data'

    # Clone GitHub Repo using Colab Secrets
    if not os.path.exists('/content/bitesafe'):
        print("Cloning private repository...")
        from google.colab import userdata

        try:
            # Fetch the token we saved in the Secrets tab
            # Make sure you name the secret exactly 'GITHUB_TOKEN'
            GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')

            # The secure URL with the injected token
            repo_url = f"https://{GITHUB_TOKEN}@github.com/miguelloeza210/dl-group-project.git"

            !git clone {repo_url} /content/bitesafe > /dev/null 2>&1
            print("Successfully cloned repository!")

        except userdata.SecretNotFoundError:
            print("ERROR: Please add 'GITHUB_TOKEN' to your Colab Secrets tab (the key icon on the left)!")
            sys.exit("Stopping execution: Missing GitHub Token.")

    # Move working directory into the cloned repo
    %cd /content/bitesafe
    CODE_DIR = '/content/bitesafe'

    if os.path.exists('requirements.txt'):
        !pip install -q -r requirements.txt

else:
    print("Running locally. Using local file system...")
    # When running locally, we assumes you already cloned
    # the repo via VS Code and have your datasets saved somewhere locally.
    CODE_DIR = os.path.abspath('.')

    # UPDATE THIS to where you keep the heavy datasets on your local PC
    ASSETS_DIR = os.path.join(CODE_DIR, 'data')

print("-" * 30)
print(f"Code Directory set to: {CODE_DIR}")
print(f"Assets Directory set to: {ASSETS_DIR}")

if os.path.exists(ASSETS_DIR):
    print(f"Success: Found data directory at {ASSETS_DIR}")
    print(f"Contents: {os.listdir(ASSETS_DIR)}")
else:
    print(f"Error: Data directory NOT found at {ASSETS_DIR}")
    print("Check if the folder is named correctly in Drive or if you need to add a shortcut.")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

if device.type == 'cuda':
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: Running on CPU. Training will be extremely slow.")

Configs

In [ ]:
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from transformers import ViTModel, ViTFeatureExtractor

CONFIG = {
    'batch_size': 16,
    'learning_rate': 1e-4,
    'num_epochs': 5,
    'embed_dim': 256,
    'margin': 1.0,                # Required for your Triplet Margin Loss
    'use_subset': True,           # Highly recommended for MVP testing
    'subset_size': 10000,         # Size of the testing subset

    # Use ASSETS_DIR for heavy files (Google Drive in Colab, Local Folder for you)
    'data_dir': os.path.join(ASSETS_DIR, 'datasets'),
    'model_save_path': os.path.join(ASSETS_DIR, 'checkpoints')
}

os.makedirs(CONFIG['model_save_path'], exist_ok=True)
print("Configuration loaded.")

Datasets & DataLoader

In [ ]:
class Recipe1MSubset(Dataset):
    def __init__(self, data_dir, is_train=True, subset_size=10000):
        # TODO: Miguel - Implement JSON/Image loading logic here
        # 1. Load Recipe1M metadata (filter for specific allergens if possible)
        # 2. Slice the dataset if subset_size is defined
        print(f"Loading {'training' if is_train else 'validation'} data...")
        self.data = [] # Placeholder for the filtered subset

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        # TODO: Miguel - Return (Image, Text_Tokens, Label/ID)
        pass

# Instantiate DataLoaders
# train_dataset = Recipe1MSubset(CONFIG['data_dir'], is_train=True, subset_size=CONFIG['subset_size'])
# train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True)
print("Data pipeline ready for implementation.")

Bi-encoder architecture

In [ ]:
class BiteSafeBiEncoder(nn.Module):
    def __init__(self, embed_dim=256):
        super(BiteSafeBiEncoder, self).__init__()

        # 1. Vision Pipeline (ViT)
        self.vit = ViTModel.from_pretrained('google/vit-base-patch16-224-in21k')
        vit_hidden_size = self.vit.config.hidden_size # Usually 768
        self.vision_projection = nn.Linear(vit_hidden_size, embed_dim)

        # 2. Text Pipeline (LSTM Baseline)
        # TODO: Miguel/Raymond - Replace with actual LSTM or BERT
        vocab_size = 10000
        lstm_hidden = 128
        self.embedding = nn.Embedding(vocab_size, 256)
        self.lstm = nn.LSTM(256, lstm_hidden, batch_first=True)
        self.text_projection = nn.Linear(lstm_hidden, embed_dim)

    def forward_vision(self, pixel_values):
        outputs = self.vit(pixel_values=pixel_values)
        # Use the CLS token representation
        cls_token = outputs.last_hidden_state[:, 0, :]
        return self.vision_projection(cls_token)

    def forward_text(self, input_ids):
        embedded = self.embedding(input_ids)
        _, (hidden, _) = self.lstm(embedded)
        # Use the final hidden state
        return self.text_projection(hidden[-1])

model = BiteSafeBiEncoder(embed_dim=CONFIG['embed_dim']).to(device)
print("Bi-Encoder model initialized and moved to:", device)

Training Loop

In [ ]:
optimizer = optim.AdamW(model.parameters(), lr=CONFIG['learning_rate'])
criterion = nn.TripletMarginLoss(margin=CONFIG['margin'], p=2)

def train_one_epoch(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0

    # TODO: Implement actual training loop over dataloader
    # Example structure:
    # for batch in dataloader:
    #     optimizer.zero_grad()
    #
    #     # 1. Extract Anchor (Image), Positive (Correct Recipe), Negative (Wrong Recipe)
    #     anchor_img, positive_text, negative_text = batch
    #
    #     # 2. Forward Passes
    #     anchor_embed = model.forward_vision(anchor_img.to(device))
    #     positive_embed = model.forward_text(positive_text.to(device))
    #     negative_embed = model.forward_text(negative_text.to(device))
    #
    #     # 3. Calculate Triplet Loss
    #     loss = criterion(anchor_embed, positive_embed, negative_embed)
    #
    #     # 4. Backward & Optimize
    #     loss.backward()
    #     optimizer.step()
    #
    #     total_loss += loss.item()

    return total_loss # / len(dataloader)

print("Training logic skeleton ready.")